# Argus AI — Phase 2: Landslide Classification (ResNet)
Dataset: bairuigong/bijie-landslide-dataset (Kaggle) — the Bijie Landslide Dataset, an academic benchmark.
Both classes (landslide / non-landslide) are cropped from the SAME TripleSat satellite imagery, same region, same time window — no domain-gap risk.

Run all cells top to bottom. Use GPU runtime: Runtime > Change runtime type > T4 GPU

This notebook saves checkpoints to Google Drive from the start and auto-resumes if disconnected.

In [ ]:
# 1. Install dependencies
!pip install kagglehub torch torchvision scikit-learn -q

In [ ]:
# 2. Mount Google Drive FIRST for checkpointing
from google.colab import drive
drive.mount('/content/drive')

import os
checkpoint_dir = '/content/drive/MyDrive/argus_ai_checkpoints/landslide_classifier'
os.makedirs(checkpoint_dir, exist_ok=True)
print("Checkpoint dir ready:", checkpoint_dir)

In [ ]:
# 3. Kaggle API setup
# Go to kaggle.com -> Settings -> API Tokens -> Create Legacy API Key -> downloads kaggle.json
from google.colab import files
uploaded = files.upload()  # select kaggle.json when prompted

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
# 4. Download dataset
import kagglehub
path = kagglehub.dataset_download("bairuigong/bijie-landslide-dataset")
print("Path to dataset files:", path)

In [ ]:
# 5. Inspect dataset structure — confirm exact folder paths before continuing
!find {path} -maxdepth 4 -type d

In [ ]:
# 6. Build train/val split (85/15) from the 'image' subfolders only (RGB images, not dem)
# NOTE: confirm these paths match cell 5's output before running — adjust if folder names differ
import random
from pathlib import Path

random.seed(42)

landslide_dir = os.path.join(path, 'Bijie-landslide-dataset', 'landslide', 'image')
nonlandslide_dir = os.path.join(path, 'Bijie-landslide-dataset', 'non-landslide', 'image')

print("Landslide dir:", landslide_dir)
print("Non-landslide dir:", nonlandslide_dir)

landslide_files = [str(p) for p in Path(landslide_dir).glob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
nonlandslide_files = [str(p) for p in Path(nonlandslide_dir).glob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]

print(f"Found {len(landslide_files)} landslide images, {len(nonlandslide_files)} non-landslide images")

random.shuffle(landslide_files)
random.shuffle(nonlandslide_files)

split_landslide = int(0.85 * len(landslide_files))
split_nonlandslide = int(0.85 * len(nonlandslide_files))

train_files = [(f, 1) for f in landslide_files[:split_landslide]] + [(f, 0) for f in nonlandslide_files[:split_nonlandslide]]
val_files = [(f, 1) for f in landslide_files[split_landslide:]] + [(f, 0) for f in nonlandslide_files[split_nonlandslide:]]

random.shuffle(train_files)
random.shuffle(val_files)

print(f"Train: {len(train_files)} images | Val: {len(val_files)} images")
print(f"Train landslide: {split_landslide}, Train non-landslide: {split_nonlandslide}")

In [ ]:
# 7. Visual sanity check — confirm both classes look like matched satellite imagery before training
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    img = Image.open(landslide_files[i]).convert('RGB')
    axes[0, i].imshow(img)
    axes[0, i].set_title('Landslide')
    axes[0, i].axis('off')

    img = Image.open(nonlandslide_files[i]).convert('RGB')
    axes[1, i].imshow(img)
    axes[1, i].set_title('Non-Landslide')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()
print("STOP: visually confirm both rows look like the same kind of satellite imagery before proceeding.")

In [ ]:
# 8. Dataset class + dataloaders (num_workers=0 to avoid Colab CPU-mode hangs)
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

class LandslideDataset(Dataset):
    def __init__(self, files, transform=None):
        self.files = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path, label = self.files[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224))
        if self.transform:
            img = self.transform(img)
        return img, label

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),  # satellite imagery has no fixed "up" — safe augmentation here
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = LandslideDataset(train_files, transform=train_transform)
val_dataset = LandslideDataset(val_files, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

print("Dataloaders ready.")

In [ ]:
# 9. Build ResNet18 model with class weighting for imbalance
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)
if device.type == 'cpu':
    print("WARNING: No GPU detected. Training will be very slow. Consider waiting for GPU quota to reset.")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)  # 2 classes: non-landslide, landslide
model = model.to(device)

num_landslide = split_landslide
num_nonlandslide = split_nonlandslide
total = num_landslide + num_nonlandslide
weight_landslide = total / (2 * num_landslide)
weight_nonlandslide = total / (2 * num_nonlandslide)
class_weights = torch.tensor([weight_nonlandslide, weight_landslide]).to(device)
print("Class weights [non-landslide, landslide]:", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

In [ ]:
# 10. Training loop with Drive checkpointing every epoch + auto-resume
import time

num_epochs = 20
best_val_acc = 0.0
start_epoch = 0

checkpoint_path = os.path.join(checkpoint_dir, 'last_checkpoint.pt')
best_model_path = os.path.join(checkpoint_dir, 'best_model.pt')

if os.path.exists(checkpoint_path):
    print("Found existing checkpoint, resuming...")
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_val_acc = ckpt['best_val_acc']
    print(f"Resuming from epoch {start_epoch}, best_val_acc so far: {best_val_acc:.4f}")

for epoch in range(start_epoch, num_epochs):
    epoch_start = time.time()

    model.train()
    running_loss, correct, total_train = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total_train += labels.size(0)

    train_loss = running_loss / total_train
    train_acc = correct / total_train

    model.eval()
    val_loss, val_correct, total_val = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            total_val += labels.size(0)

    val_loss = val_loss / total_val
    val_acc = val_correct / total_val
    scheduler.step()

    epoch_time = time.time() - epoch_start
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | Time: {epoch_time:.1f}s")

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_acc': max(best_val_acc, val_acc),
    }, checkpoint_path)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"  -> New best model saved (val_acc: {val_acc:.4f})")

print("\nTraining complete. Best val accuracy:", best_val_acc)

In [ ]:
# 11. Final evaluation — confusion matrix, precision, recall, F1
from sklearn.metrics import classification_report, confusion_matrix

model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=['Non-Landslide', 'Landslide']))
print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))
print("\nIf this shows 100.00% across every metric, STOP and flag it before proceeding — that pattern previously indicated a hidden shortcut, not genuine performance.")

In [ ]:
# 12. Quick visual inference test
test_path, test_label = val_files[0]
img = Image.open(test_path).convert('RGB')
img_tensor = val_transform(img).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    output = model(img_tensor)
    probs = torch.softmax(output, dim=1)
    pred = torch.argmax(probs, dim=1).item()

labels_map = {0: 'Non-Landslide', 1: 'Landslide'}
plt.imshow(img)
plt.title(f"True: {labels_map[test_label]} | Predicted: {labels_map[pred]} ({probs[0][pred]*100:.1f}%)")
plt.axis('off')
plt.show()

In [ ]:
# 13. Download the best model to your laptop
from google.colab import files
files.download(best_model_path)

## Next steps
1. Download `best_model.pt` from cell 13
2. Move it into your local repo: `argus-ai/models/landslide_classification/best_model.pt`
3. Note down final val accuracy, precision, recall, F1 from cell 11 for your resume/README
4. Move to Phase 3: LangGraph multi-agent layer

## If disconnected mid-training
Reconnect, rerun cells 1-9 (setup), then run cell 10 — it auto-detects the existing checkpoint and resumes automatically.